In [4]:
from __future__ import annotations
import os
from io import StringIO
from pathlib import Path
from typing import Dict, Any, List, Optional
from src.config import get_secret
import numpy as np
import pandas as pd
import requests


YEAR: int = 2026
TOUR_KEYS: List[str] = ["PGA", "EURO", "LIV"]

DG_ROUNDS_URL: str = "https://feeds.datagolf.com/historical-raw-data/rounds"
DG_FIELD_URL: str  = "https://feeds.datagolf.com/field-updates"
DG_ODDS_URL: str   = "https://feeds.datagolf.com/betting-tools/outrights"

DG_API_KEY = get_secret("DATAGOLF_API_KEY")
if not DG_API_KEY:
    raise RuntimeError("Missing DATAGOLF_API_KEY env var. Set it and rerun.")

# Project paths (absolute to your repo)
PROJECT_ROOT: Path = Path("/Users/joshmacbook/python_projects/GolfStats")
DATA_ROOT: Path    = PROJECT_ROOT / "Data"
RAW_DIR: Path      = DATA_ROOT / "Raw"
CLEAN_DIR: Path    = DATA_ROOT / "Clean" / "Combined"
INUSE_DIR: Path    = DATA_ROOT / "in Use"

# Create dirs
for _p in [RAW_DIR, CLEAN_DIR, INUSE_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

# Canonical output paths (things we DO update)
CLEAN_YEAR_PATH: Path = CLEAN_DIR / f"combined_rounds_{YEAR}.csv"
INUSE_ALL_PATH: Path  = INUSE_DIR / "combined_rounds_all_2017_2026.csv"
MST_24P_PATH: Path    = INUSE_DIR / "combined_roundlevel_2024_present.csv"
FINISHES_PATH: Path   = INUSE_DIR / "Finishes.csv"

# Weekly field/odds artifacts (PGA only for field/owgr + odds)
THIS_WEEK_FIELD_CSV: Path = INUSE_DIR / "this_week_field.csv"
THIS_WEEK_ODDS_CSV: Path  = INUSE_DIR / "this_week_odds.csv"

FIELDS_XLSX: Path       = INUSE_DIR / "Fields.xlsx"
ALL_PLAYERS_XLSX: Path  = INUSE_DIR / "All_players.xlsx"
SCHED_XLSX: Path        = INUSE_DIR / "OAD_2026_Schedule.xlsx"

# Things we explicitly DO NOT touch in the weekly refresh notebook
MST_17_23_EVENTMEAN_PATH: Path = INUSE_DIR / "combined_eventlevel_pga_2017_2023_mean.csv"

# ----------------------------
# Tour Mapping
# ----------------------------
# Keys are your internal tour keys; api_tour is what DataGolf expects.
TOURS: Dict[str, Dict[str, str]] = {
    "PGA":  {"folder": "PGA",  "prefix": "PGA",  "api_tour": "PGA"},
    "EURO": {"folder": "EURO", "prefix": "EURO", "api_tour": "euro"},
    "LIV":  {"folder": "LIV",  "prefix": "liv",  "api_tour": "liv"},
}

# ----------------------------
# Requests session
# ----------------------------
TIMEOUT: int = 60
SESSION = requests.Session()

# ----------------------------
# Small Utilities
# ----------------------------
def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def coerce_int(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df

def safe_to_datetime(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")

def pull_csv_to_df(url: str, params: Dict[str, Any], *, save_path: Optional[Path] = None) -> pd.DataFrame:
    resp = SESSION.get(url, params=params, timeout=TIMEOUT)
    resp.raise_for_status()
    df = pd.read_csv(StringIO(resp.text))
    if save_path is not None:
        ensure_dir(save_path.parent)
        df.to_csv(save_path, index=False)
        print(f"[PULL] Saved → {save_path} | rows: {len(df):,}")
    return df

def read_excel(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_excel(path)

def write_excel(path: Path, df: pd.DataFrame, *, sheet_name: str = "Sheet1") -> None:
    ensure_dir(path.parent)
    with pd.ExcelWriter(path, engine="openpyxl", mode="w") as w:
        df.to_excel(w, index=False, sheet_name=sheet_name)

def read_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, **kwargs)

def write_csv(path: Path, df: pd.DataFrame) -> None:
    ensure_dir(path.parent)
    df.to_csv(path, index=False)

# ----------------------------
# DataGolf Pullers
# ----------------------------
def pull_rounds_year(*, tour_key: str, year: int = YEAR) -> Path:
    if tour_key not in TOURS:
        raise ValueError(f"Unknown tour_key: {tour_key}. Expected one of {list(TOURS)}")

    cfg = TOURS[tour_key]
    out_dir = ensure_dir(RAW_DIR / cfg["folder"])
    out_path = out_dir / f"{cfg['prefix']}_rounds_{year}.csv"

    params = {
        "tour": cfg["api_tour"],
        "event_id": "all",
        "year": year,
        "file_format": "csv",
        "key": DG_API_KEY,
    }

    df = pull_csv_to_df(DG_ROUNDS_URL, params, save_path=out_path)
    df = coerce_int(df, ["event_id", "dg_id", "year", "round_num"])
    df.to_csv(out_path, index=False)

    print(f"[ROUNDS] {tour_key} {year} → {out_path} | rows: {len(df):,}")
    return out_path

def pull_this_week_field_pga(*, save_path: Path = THIS_WEEK_FIELD_CSV) -> pd.DataFrame:
    params = {"tour": "PGA", "file_format": "csv", "key": DG_API_KEY}
    df = pull_csv_to_df(DG_FIELD_URL, params, save_path=save_path)
    df = coerce_int(df, ["event_id", "dg_id", "year", "round_num"])
    return df

def pull_this_week_odds_pga(*, save_path: Path = THIS_WEEK_ODDS_CSV) -> pd.DataFrame:
    params = {
    "tour": "pga",
    "market": "win",
    "odds_format": "decimal",
    "file_format": "csv",
    "key": DG_API_KEY,
    }
    df = pull_csv_to_df(DG_ODDS_URL, params, save_path=save_path)
    df = coerce_int(df, ["event_id", "dg_id", "year"])
    return df

# ----------------------------
# Cleaning helpers (finish_num)
# ----------------------------
NON_NUMERIC_FINISH = {"CUT", "WD", "DQ", "MDF", "NAN"}

def clean_finish(val):
    v = str(val).upper().strip()
    if v in NON_NUMERIC_FINISH:
        return v
    if v.startswith("T") and v[1:].isdigit():
        return int(v[1:])
    if v.isdigit():
        return int(v)
    return v

# ----------------------------
# LIV patch map (event_id + course_num)
# ----------------------------
LIV_EVENT_ID_MAP_RAW = {
    "adelaide":                         1012,
    "andalucia":                        1024,
    "bangkok":                          1006,
    "bedminster":                       1003,
    "boston":                           1004,
    "chicago":                          1005,
    "dallas":                           1031,
    "dallas (team finalstroke play)":   1026,
    "dc":                               1015,
    "greenbrier":                       1017,
    "hong kong":                        1020,
    "houston":                          1022,
    "indianapolis":                     1032,
    "jeddah":                           1007,
    "korea":                            1029,
    "las vegas":                        1019,
    "london":                           1001,
    "mayakoba":                         1009,
    "mexico city":                      1028,
    "miami":                            1021,
    "miami (team finalstroke play)":    1008,
    "michigan (team finalstroke play)": 1033,
    "nashville":                        1023,
    "orlando":                          1011,
    "portland":                         1002,
    "promotions event":                 1018,
    "riyadh":                           1027,
    "singapore":                        1013,
    "tucson":                           1010,
    "tulsa":                            1014,
    "united kingdom":                   1025,
    "valderrama":                       1016,
    "virginia":                         1030,
}
LIV_EVENT_ID_MAP = {k.strip().lower(): v for k, v in LIV_EVENT_ID_MAP_RAW.items()}

def patch_liv_ids(df: pd.DataFrame) -> pd.DataFrame:
    if not {"tour", "event_name"}.issubset(df.columns):
        return df

    out = df.copy()
    out["tour_clean"] = out["tour"].astype(str).str.strip().str.upper()
    out["event_name_clean"] = out["event_name"].astype(str).str.strip().str.lower()

    liv_mask = out["tour_clean"].eq("LIV")
    in_map = out["event_name_clean"].isin(LIV_EVENT_ID_MAP.keys())
    target = liv_mask & in_map

    if target.any():
        new_ids = out.loc[target, "event_name_clean"].map(LIV_EVENT_ID_MAP).astype("Int64")
        if "event_id" in out.columns:
            out.loc[target, "event_id"] = new_ids
        if "course_num" in out.columns:
            out.loc[target, "course_num"] = new_ids
        print(f"[PATCH] LIV id/course_num updates: {int(target.sum()):,}")

    return out.drop(columns=["tour_clean", "event_name_clean"], errors="ignore")

# ----------------------------
# round_date derivation from event_completed + round_num
# ----------------------------
def add_round_date(df: pd.DataFrame) -> pd.DataFrame:
    needed = {"event_completed", "round_num", "tour", "event_name"}
    if not needed.issubset(df.columns):
        return df

    out = df.copy()
    out["event_completed_dt"] = safe_to_datetime(out["event_completed"])
    if out["event_completed_dt"].isna().all():
        return df

    out["round_num"] = pd.to_numeric(out["round_num"], errors="coerce")
    group_cols = ["tour", "event_name", "event_completed_dt"]
    max_round = out.groupby(group_cols)["round_num"].transform("max")
    offset_days = max_round - out["round_num"]

    out["round_date_dt"] = out["event_completed_dt"] - pd.to_timedelta(offset_days, unit="D")
    out["round_date"] = out["round_date_dt"].dt.date.astype(str)

    return out.drop(columns=["event_completed_dt", "round_date_dt"], errors="ignore")

# ----------------------------
# Safe replace helpers (surgical 2026-only updates)
# ----------------------------
def replace_year_in_csv(existing_path: Path, new_year_df: pd.DataFrame, *, year_col: str = "year", year_val: int = YEAR) -> None:
    if existing_path.exists():
        old = pd.read_csv(existing_path)
        old[year_col] = pd.to_numeric(old.get(year_col), errors="coerce")
        kept = old[old[year_col] != year_val].copy()
        out = pd.concat([kept, new_year_df], ignore_index=True)
        out.to_csv(existing_path, index=False)
        print(f"[REPLACE] {existing_path} | dropped {year_val}, appended {year_val} | rows: {len(out):,}")
    else:
        new_year_df.to_csv(existing_path, index=False)
        print(f"[WRITE] {existing_path} | rows: {len(new_year_df):,}")

# ----------------------------
# Finishes builder (2026-only)
# ----------------------------
def build_finishes_2026_from_roundlevel(roundlevel_24p_path: Path = MST_24P_PATH, out_path: Path = FINISHES_PATH) -> None:
    df = pd.read_csv(roundlevel_24p_path)

    # derive year
    if "season" in df.columns:
        df["year"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
    else:
        if "round_date" not in df.columns:
            raise RuntimeError("Cannot derive year: no 'season' and no 'round_date' in roundlevel file.")
        df["round_date"] = pd.to_datetime(df["round_date"], errors="coerce")
        df["year"] = df["round_date"].dt.year.astype("Int64")

    df = df[df["year"] == YEAR].copy()

    df["event_id"] = pd.to_numeric(df.get("event_id"), errors="coerce").astype("Int64")
    df["dg_id"]    = pd.to_numeric(df.get("dg_id"), errors="coerce").astype("Int64")
    df["finish_num"] = pd.to_numeric(df.get("finish_num"), errors="coerce")

    sort_cols = [c for c in ["year", "event_id", "dg_id", "round_num", "round_date"] if c in df.columns]
    df_sorted = df.sort_values(sort_cols)

    final = (
        df_sorted
        .groupby(["year", "event_id", "dg_id"], as_index=False)
        .tail(1)
        .copy()
    )

    final["made_cut"] = final["finish_num"].notna().astype(int)
    final["CUT"] = (1 - final["made_cut"]).astype(int)

    fn = final["finish_num"]
    final["win"]    = ((fn == 1)  & fn.notna()).astype(int)
    final["top_5"]  = ((fn <= 5)  & fn.notna()).astype(int)
    final["top_10"] = ((fn <= 10) & fn.notna()).astype(int)
    final["top_25"] = ((fn <= 25) & fn.notna()).astype(int)

    if "finish_text" not in final.columns and "fin_text" in final.columns:
        final["finish_text"] = final["fin_text"]

    cols = [
        "year", "event_name", "event_id", "event_completed",
        "player_name", "dg_id", "finish_text", "finish_num",
        "win", "top_5", "top_10", "top_25", "made_cut", "CUT"
    ]
    for c in cols:
        if c not in final.columns:
            final[c] = pd.NA

    out = final[cols].sort_values(["event_id", "finish_num", "player_name"], na_position="last")
    out.to_csv(out_path, index=False)
    print(f"[FINISHES] Wrote {len(out):,} rows for {YEAR} → {out_path}")


In [6]:
# ============================================================
# BLOCK 1: Weekly PGA Field + Odds
#   - Field membership: authoritative snapshot (overwrite per event)
#   - close_odds: updatable only before event start; frozen once started
# ============================================================

from datetime import datetime
import pytz

ET = pytz.timezone("America/New_York")
NOW_ET = pd.Timestamp.now(tz=ET)

def _schedule_start_dt_et(sched_df: pd.DataFrame, event_id: int) -> pd.Timestamp:
    row = sched_df.loc[sched_df["event_id"] == event_id]
    if row.empty:
        raise RuntimeError(f"Schedule has no row for event_id={event_id}")

    start_date_value = pd.to_datetime(row["start_date"].iloc[0], errors="coerce")
    if pd.isna(start_date_value):
        raise RuntimeError(f"Schedule start_date is NaT for event_id={event_id}")

    # Treat start as midnight ET on start_date
    start_dt = pd.Timestamp(start_date_value.date())
    start_dt = start_dt.tz_localize(ET)
    return start_dt

def _ensure_fields_cols(df: pd.DataFrame) -> pd.DataFrame:
    # Required schema for Fields.xlsx
    required_cols = ["year", "event_name", "event_id", "event_completed", "player_name", "dg_id", "close_odds"]
    out = df.copy()
    for c in required_cols:
        if c not in out.columns:
            out[c] = pd.NA

    # Optional audit cols (recommended)
    for c in ["field_pulled_at", "odds_pulled_at"]:
        if c not in out.columns:
            out[c] = pd.NA

    out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")
    out["event_id"] = pd.to_numeric(out["event_id"], errors="coerce").astype("Int64")
    out["dg_id"] = pd.to_numeric(out["dg_id"], errors="coerce").astype("Int64")
    return out

# -------------------------
# 1) Pull this week field (PGA)
# -------------------------
field_raw = pull_this_week_field_pga(save_path=THIS_WEEK_FIELD_CSV)  # <-- fixed (removed stray "a")

need_field = ["event_name", "event_id", "dg_id", "player_name", "owgr_rank"]
missing = [c for c in need_field if c not in field_raw.columns]
if missing:
    raise RuntimeError(f"Field feed missing columns: {missing}. Got: {list(field_raw.columns)}")

field = field_raw[need_field].copy()
field = coerce_int(field, ["event_id", "dg_id", "year", "round_num"])
field["owgr_rank"] = pd.to_numeric(field["owgr_rank"], errors="coerce")
field = field.dropna(subset=["event_id", "dg_id", "player_name"]).copy()
field["player_name"] = field["player_name"].astype(str)
field["event_name"]  = field["event_name"].astype(str)

field_event_ids = sorted(field["event_id"].dropna().unique().tolist())
if len(field_event_ids) != 1:
    raise RuntimeError(f"Expected exactly 1 event_id in field feed; got {field_event_ids}")

FIELD_EVENT_ID = int(field_event_ids[0])
FIELD_EVENT_NAME = field["event_name"].iloc[0]
print(f"[FIELD] event_id={FIELD_EVENT_ID} | event_name={FIELD_EVENT_NAME} | players={len(field):,}")

# -------------------------
# 2) Pull this week odds (PGA)
#   - always save the file so you can inspect “next event” mid-event
#   - only write close_odds into Fields.xlsx if:
#       a) we can identify a single ODDS_EVENT_ID
#       b) the event has NOT started yet (freeze once started)
#       c) we already have a field block for that event in Fields.xlsx (don’t create members)
# -------------------------
try:
    odds_raw = pull_this_week_odds_pga(save_path=THIS_WEEK_ODDS_CSV)
except Exception as e:
    if THIS_WEEK_ODDS_CSV.exists():
        print("[ODDS] pull failed; falling back to existing file:", THIS_WEEK_ODDS_CSV)
        odds_raw = pd.read_csv(THIS_WEEK_ODDS_CSV)
    else:
        raise RuntimeError(f"[ODDS] pull failed and no existing file found. Error: {e}") from e

odds_raw = coerce_int(odds_raw, ["event_id", "dg_id", "year"])

if "dg_id" not in odds_raw.columns:
    raise RuntimeError(f"Odds missing dg_id. Got columns: {list(odds_raw.columns)}")
if "datagolf_base_history_fit" not in odds_raw.columns:
    raise RuntimeError(
        "Odds missing datagolf_base_history_fit (needed for close_odds). "
        f"Got columns: {list(odds_raw.columns)}"
    )

# Identify which event these odds belong to (must be unambiguous)
ODDS_EVENT_ID = None
if "event_id" in odds_raw.columns:
    odds_event_ids = sorted(pd.to_numeric(odds_raw["event_id"], errors="coerce").dropna().astype(int).unique().tolist())
    if len(odds_event_ids) == 1:
        ODDS_EVENT_ID = int(odds_event_ids[0])
    else:
        print(f"[ODDS] ambiguous event_id(s) in odds feed: {odds_event_ids}. Will NOT write close_odds into Fields.xlsx.")
else:
    print("[ODDS] no event_id column in odds feed. Will NOT write close_odds into Fields.xlsx.")

odds = odds_raw[["dg_id", "datagolf_base_history_fit"]].copy()
odds["dg_id"] = pd.to_numeric(odds["dg_id"], errors="coerce").astype("Int64")
odds["datagolf_base_history_fit"] = pd.to_numeric(odds["datagolf_base_history_fit"], errors="coerce")
odds = odds.dropna(subset=["dg_id"]).copy()

close_odds_by_id = dict(zip(odds["dg_id"].astype(int), odds["datagolf_base_history_fit"]))
print(f"[ODDS] rows={len(odds):,} | mapped_ids={len(close_odds_by_id):,} | ODDS_EVENT_ID={ODDS_EVENT_ID}")

# -------------------------
# 3) Load schedule
# -------------------------
sched = read_excel(SCHED_XLSX)
for c in ["event_id", "start_date"]:
    if c not in sched.columns:
        raise RuntimeError(f"Schedule missing '{c}'. Got columns: {list(sched.columns)}")

sched["event_id"] = pd.to_numeric(sched["event_id"], errors="coerce").astype("Int64")
sched["start_date"] = pd.to_datetime(sched["start_date"], errors="coerce")

# Field event start dt (ET)
FIELD_START_DT_ET = _schedule_start_dt_et(sched, FIELD_EVENT_ID)
print(f"[SCHED] FIELD_EVENT_ID={FIELD_EVENT_ID} start_dt_et={FIELD_START_DT_ET} now_et={NOW_ET}")

# -------------------------
# 4) Update Fields.xlsx
#   - Always overwrite membership for FIELD_EVENT_ID (authoritative snapshot)
#   - close_odds: only update for a given event before it starts; otherwise preserve
# -------------------------
fields_existing = read_excel(FIELDS_XLSX)
fields_existing = _ensure_fields_cols(fields_existing)

# --- Build authoritative weekly snapshot for FIELD_EVENT_ID ---
wk = field[["event_name", "event_id", "dg_id", "player_name"]].copy()
wk["year"] = YEAR
wk["event_completed"] = pd.Timestamp(FIELD_START_DT_ET.date())  # keep as date-like value in file
wk["field_pulled_at"] = NOW_ET.strftime("%Y-%m-%d %H:%M:%S %Z")

# For FIELD_EVENT_ID, decide whether to update close_odds or preserve existing (freeze)
mask_field_block = (fields_existing["year"] == YEAR) & (fields_existing["event_id"] == FIELD_EVENT_ID)
existing_block = fields_existing.loc[mask_field_block, ["dg_id", "close_odds"]].copy()
existing_block["dg_id"] = pd.to_numeric(existing_block["dg_id"], errors="coerce").astype("Int64")
existing_odds_map = dict(zip(existing_block["dg_id"].dropna().astype(int), existing_block["close_odds"]))

field_started = NOW_ET >= FIELD_START_DT_ET
if field_started:
    print("[FIELDS] FIELD_EVENT started → freezing close_odds for FIELD_EVENT (preserving existing values).")
    wk["close_odds"] = wk["dg_id"].astype(int).map(existing_odds_map)
    wk["odds_pulled_at"] = pd.NA
else:
    print("[FIELDS] FIELD_EVENT not started → updating close_odds for FIELD_EVENT from latest odds feed.")
    wk["close_odds"] = wk["dg_id"].astype(int).map(close_odds_by_id)
    wk["odds_pulled_at"] = NOW_ET.strftime("%Y-%m-%d %H:%M:%S %Z")

# Normalize schema
wk = wk[["year", "event_name", "event_id", "event_completed", "player_name", "dg_id", "close_odds", "field_pulled_at", "odds_pulled_at"]].copy()

# HARD REPLACE membership for FIELD_EVENT_ID
dropped = int(mask_field_block.sum())
fields_new = pd.concat([fields_existing.loc[~mask_field_block].copy(), wk], ignore_index=True)
fields_new["player_name"] = fields_new["player_name"].astype(str)
fields_new["event_name"]  = fields_new["event_name"].astype(str)

write_excel(FIELDS_XLSX, fields_new)
print(f"[FIELDS] FIELD_EVENT snapshot written. dropped_old_rows={dropped:,} | week={len(wk):,} | final={len(fields_new):,}")

# --- Optional: Update odds for ODDS_EVENT_ID (NEXT event) if safe ---
# Only if:
#   1) ODDS_EVENT_ID is known and exists in schedule
#   2) event has NOT started
#   3) Fields.xlsx already has a membership block for that event (no creating members)
if ODDS_EVENT_ID is not None and ODDS_EVENT_ID != FIELD_EVENT_ID:
    try:
        ODDS_START_DT_ET = _schedule_start_dt_et(sched, ODDS_EVENT_ID)
        odds_started = NOW_ET >= ODDS_START_DT_ET

        mask_odds_block = (fields_new["year"] == YEAR) & (fields_new["event_id"] == ODDS_EVENT_ID)
        has_block = bool(mask_odds_block.any())

        print(f"[ODDS->FIELDS] ODDS_EVENT_ID={ODDS_EVENT_ID} start_dt_et={ODDS_START_DT_ET} started={odds_started} has_fields_block={has_block}")

        if odds_started:
            print("[ODDS->FIELDS] Event already started → NOT updating close_odds (frozen).")
        elif not has_block:
            print("[ODDS->FIELDS] No field block exists for this event yet → NOT writing close_odds (won't create members).")
        else:
            # Update close_odds only for existing members in that block
            tmp = fields_new.copy()
            upd = tmp.loc[mask_odds_block, "dg_id"].astype("Int64").dropna().astype(int)
            tmp.loc[mask_odds_block, "close_odds"] = tmp.loc[mask_odds_block, "dg_id"].astype("Int64").map(
                lambda x: close_odds_by_id.get(int(x), pd.NA) if pd.notna(x) else pd.NA
            )
            tmp.loc[mask_odds_block, "odds_pulled_at"] = NOW_ET.strftime("%Y-%m-%d %H:%M:%S %Z")

            write_excel(FIELDS_XLSX, tmp)
            fields_new = tmp
            print("[ODDS->FIELDS] Updated close_odds for ODDS_EVENT members (pre-start).")
    except Exception as e:
        print(f"[ODDS->FIELDS] Skipping odds write due to error: {e}")

# -------------------------
# 5) Update All_players.xlsx owgr for dg_ids in this week field
# -------------------------
ap = read_excel(ALL_PLAYERS_XLSX)
if "dg_id" not in ap.columns:
    raise RuntimeError(f"All_players.xlsx missing dg_id. Got columns: {list(ap.columns)}")
if "owgr" not in ap.columns:
    ap["owgr"] = pd.NA

ap["dg_id"] = pd.to_numeric(ap["dg_id"], errors="coerce").astype("Int64")

owgr_map = dict(zip(field["dg_id"].dropna().astype(int), field["owgr_rank"]))

before_nonnull = ap["owgr"].notna().sum()
ap["owgr"] = ap.apply(
    lambda r: owgr_map.get(int(r["dg_id"]), r["owgr"]) if pd.notna(r["dg_id"]) else r["owgr"],
    axis=1,
)
after_nonnull = ap["owgr"].notna().sum()

write_excel(ALL_PLAYERS_XLSX, ap)
print(f"[ALL_PLAYERS] owgr updated. nonnull_before={before_nonnull:,} | nonnull_after={after_nonnull:,} | updated_ids={len(owgr_map):,}")

[PULL] Saved → /Users/joshmacbook/python_projects/GolfStats/Data/in Use/this_week_field.csv | rows: 72
[FIELD] event_id=9 | event_name=Arnold Palmer Invitational presented by Mastercard | players=72
[PULL] Saved → /Users/joshmacbook/python_projects/GolfStats/Data/in Use/this_week_odds.csv | rows: 72
[ODDS] no event_id column in odds feed. Will NOT write close_odds into Fields.xlsx.
[ODDS] rows=72 | mapped_ids=72 | ODDS_EVENT_ID=None
[SCHED] FIELD_EVENT_ID=9 start_dt_et=2026-03-05 00:00:00-05:00 now_et=2026-03-07 09:53:37.852153-05:00
[FIELDS] FIELD_EVENT started → freezing close_odds for FIELD_EVENT (preserving existing values).
[FIELDS] FIELD_EVENT snapshot written. dropped_old_rows=72 | week=72 | final=1,405
[ALL_PLAYERS] owgr updated. nonnull_before=199 | nonnull_after=199 | updated_ids=72


In [10]:
# ============================================================
# BLOCK 2: Weekly rounds refresh (2026 only) for PGA/EURO/LIV
# ============================================================

print("\n[GUARD] NOT touching:", MST_17_23_EVENTMEAN_PATH)

# 1) Pull 2026 rounds for each tour into Raw
for tour_key in TOUR_KEYS:
    pull_rounds_year(tour_key=tour_key, year=YEAR)

# 2) Load only the freshly pulled 2026 files and stack
dfs = []
for tour_key in TOUR_KEYS:
    cfg = TOURS[tour_key]
    fpath = RAW_DIR / cfg["folder"] / f"{cfg['prefix']}_rounds_{YEAR}.csv"
    df = read_csv(fpath)
    df["tour"] = tour_key
    df["year"] = YEAR
    dfs.append(df)

combined_2026 = pd.concat(dfs, ignore_index=True)
combined_2026 = coerce_int(combined_2026, ["event_id", "dg_id", "year", "round_num"])

# 3) finish_num from fin_text (if present)
if "fin_text" in combined_2026.columns:
    combined_2026["fin_text"] = combined_2026["fin_text"].astype(str).str.upper().str.strip()
    combined_2026["finish_num"] = combined_2026["fin_text"].apply(clean_finish)
    print("[CLEAN] finish_num created from fin_text")
else:
    print("[CLEAN] fin_text not present; skipping finish_num creation")

# 4) Patch LIV event_id/course_num
combined_2026 = patch_liv_ids(combined_2026)

# Drop event_id 18 (unwanted event)
combined_2026 = combined_2026[combined_2026["event_id"] != 18].copy()
print(f"[FILTER] Dropped event_id=18 | rows remaining: {len(combined_2026):,}")

# 5) round_date derivation
combined_2026 = add_round_date(combined_2026)

# 6) Write Clean/Combined/combined_rounds_2026.csv
write_csv(CLEAN_YEAR_PATH, combined_2026)
print(f"[CLEAN] wrote {CLEAN_YEAR_PATH} | rows={len(combined_2026):,}")

# 7) Replace 2026 in the big combined file
replace_year_in_csv(INUSE_ALL_PATH, combined_2026, year_col="year", year_val=YEAR)

# 8) Run add_round_positions on the full combined file
#    Must run on the full file — positions are calculated across the entire field per event
def add_round_positions(df: pd.DataFrame) -> pd.DataFrame:
    for col in ["to_par", "cum_to_par", "round_position", "round_position_text"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    df = df.copy()
    df["to_par"] = df["round_score"] - df["course_par"]

    df = df.sort_values(["event_id", "season", "dg_id", "round_num"]).reset_index(drop=True)
    df["cum_to_par"] = df.groupby(["event_id", "season", "dg_id"])["to_par"].cumsum()

    # Fallback for events missing course_par: rank on raw cumulative score
    missing_par_events = df.groupby(["event_id", "season"])["course_par"].apply(lambda x: x.isna().all())
    missing_par_keys = missing_par_events[missing_par_events].index
    if len(missing_par_keys) > 0:
        mask = df.set_index(["event_id", "season"]).index.isin(missing_par_keys)
        df.loc[mask, "cum_to_par"] = (
            df[mask]
            .groupby(["event_id", "season", "dg_id"])["round_score"]
            .cumsum()
            .values
        )

    # Rank players within each tournament + round (include_groups=False suppresses FutureWarning)
    def rank_group(group):
        valid = group["cum_to_par"].notna()
        positions = pd.Series(np.nan, index=group.index)
        if valid.any():
            positions[valid] = (
                group.loc[valid, "cum_to_par"]
                .rank(method="min", ascending=True)
                .astype(int)
            )
        return positions

    df["round_position"] = (
        df.groupby(["event_id", "season", "round_num"], group_keys=False)
        .apply(lambda g: rank_group(g), include_groups=False)
    )

    df["round_position"] = pd.array(
        df["round_position"].where(df["round_position"].isna(), df["round_position"].astype("Int64")),
        dtype="Int64",
    )

    tie_counts = df.groupby(["event_id", "season", "round_num", "round_position"])["dg_id"].transform("count")
    df["round_position_text"] = df["round_position"].astype(str)
    df.loc[df["round_position"].isna(), "round_position_text"] = np.nan
    df.loc[tie_counts > 1, "round_position_text"] = "T" + df.loc[tie_counts > 1, "round_position"].astype(str)

    return df

print(f"[POSITIONS] Running add_round_positions on full file ({INUSE_ALL_PATH.name})...")
full_df = read_csv(INUSE_ALL_PATH, low_memory=False)
full_df = add_round_positions(full_df)

nan_count = full_df["round_position"].isna().sum()
if nan_count > 0:
    nan_events = full_df[full_df["round_position"].isna()]["event_name"].value_counts()
    print(f"  {nan_count:,} rows with NaN round_position:")
    for event, count in nan_events.items():
        print(f"    {event}: {count} rows")

write_csv(INUSE_ALL_PATH, full_df)
print(f"[POSITIONS] Done. {len(full_df):,} rows, {len(full_df.columns)} columns")

# 9) Build Finishes.csv (2026 only) — sourced from the full file, filtered to 2026
build_finishes_2026_from_roundlevel(INUSE_ALL_PATH, FINISHES_PATH)

print("[DONE] weekly rounds refresh complete.")


[GUARD] NOT touching: /Users/joshmacbook/python_projects/GolfStats/Data/in Use/combined_eventlevel_pga_2017_2023_mean.csv
[PULL] Saved → /Users/joshmacbook/python_projects/GolfStats/Data/Raw/PGA/PGA_rounds_2026.csv | rows: 2,699
[ROUNDS] PGA 2026 → /Users/joshmacbook/python_projects/GolfStats/Data/Raw/PGA/PGA_rounds_2026.csv | rows: 2,699
[PULL] Saved → /Users/joshmacbook/python_projects/GolfStats/Data/Raw/EURO/EURO_rounds_2026.csv | rows: 2,345
[ROUNDS] EURO 2026 → /Users/joshmacbook/python_projects/GolfStats/Data/Raw/EURO/EURO_rounds_2026.csv | rows: 2,345
[PULL] Saved → /Users/joshmacbook/python_projects/GolfStats/Data/Raw/LIV/liv_rounds_2026.csv | rows: 607
[ROUNDS] LIV 2026 → /Users/joshmacbook/python_projects/GolfStats/Data/Raw/LIV/liv_rounds_2026.csv | rows: 607
[CLEAN] finish_num created from fin_text
[PATCH] LIV id/course_num updates: 607
[FILTER] Dropped event_id=18 | rows remaining: 5,651
[CLEAN] wrote /Users/joshmacbook/python_projects/GolfStats/Data/Clean/Combined/combine

/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_59995/905869689.py:261: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  old = pd.read_csv(existing_path)


[REPLACE] /Users/joshmacbook/python_projects/GolfStats/Data/in Use/combined_rounds_all_2017_2026.csv | dropped 2026, appended 2026 | rows: 316,691
[POSITIONS] Running add_round_positions on full file (combined_rounds_all_2017_2026.csv)...
  924 rows with NaN round_position:
    Zurich Classic of New Orleans: 924 rows
[POSITIONS] Done. 316,691 rows, 41 columns
[FINISHES] Wrote 1,935 rows for 2026 → /Users/joshmacbook/python_projects/GolfStats/Data/in Use/Finishes.csv
[DONE] weekly rounds refresh complete.


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_59995/905869689.py:275: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(roundlevel_24p_path)


In [11]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from src.config import get_secret
# ──────────────────────────────────────────────────────────────────────────────

DG_API_KEY = get_secret("DATAGOLF_API_KEY")
PERIODS = ["l12", "l24", "ytd"]
BASE_URL = "https://feeds.datagolf.com/preds/approach-skill"
OUTPUT_PATH = Path("/Users/joshmacbook/python_projects/GolfStats/Data/in Use/approach_skill_all_periods.csv")

def fetch_period(period: str) -> pd.DataFrame:
    """Fetch approach-skill data for a single time period."""
    params = {
        "period": period,
        "file_format": "json",
        "key": DG_API_KEY,
    }

    resp = requests.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()

    payload = resp.json()

    # The API returns a dict with a players list + metadata at the top level.
    # Handle both shapes: list directly, or nested under a key.
    if isinstance(payload, list):
        rows = payload
        last_updated = None
    else:
        # Try common key names
        rows = (
            payload.get("players")
            or payload.get("data")
            or payload.get("results")
            or []
        )
        last_updated = payload.get("last_updated") or payload.get("updated_at")

    df = pd.DataFrame(rows)

    # Tag with metadata
    df["time_period"] = period
    df["last_updated"] = last_updated
    df["fetched_at"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    return df


def main():
    all_frames = []

    for period in PERIODS:
        print(f"Fetching period: {period} ...", end=" ")
        try:
            df = fetch_period(period)
            print(f"{len(df)} rows")
            all_frames.append(df)
        except requests.HTTPError as e:
            print(f"HTTP error — {e}")
        except Exception as e:
            print(f"Failed — {e}")

    if not all_frames:
        print("No data fetched. Exiting.")
        return

    combined = pd.concat(all_frames, ignore_index=True)

    # Reorder: metadata columns first, then everything else
    meta_cols = ["time_period", "last_updated", "fetched_at"]
    other_cols = [c for c in combined.columns if c not in meta_cols]
    combined = combined[meta_cols + other_cols]

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    combined.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved {len(combined)} total rows -> {OUTPUT_PATH}")
    print(f"Columns: {list(combined.columns)}")


if __name__ == "__main__":
    main()

Fetching period: l12 ... 325 rows
Fetching period: l24 ... 330 rows
Fetching period: ytd ... 186 rows

Saved 841 total rows -> /Users/joshmacbook/python_projects/GolfStats/Data/in Use/approach_skill_all_periods.csv
Columns: ['time_period', 'last_updated', 'fetched_at', '100_150_fw_gir_rate', '100_150_fw_good_shot_rate', '100_150_fw_low_data_indicator', '100_150_fw_poor_shot_avoid_rate', '100_150_fw_proximity_per_shot', '100_150_fw_sg_per_shot', '100_150_fw_shot_count', '150_200_fw_gir_rate', '150_200_fw_good_shot_rate', '150_200_fw_low_data_indicator', '150_200_fw_poor_shot_avoid_rate', '150_200_fw_proximity_per_shot', '150_200_fw_sg_per_shot', '150_200_fw_shot_count', '50_100_fw_gir_rate', '50_100_fw_good_shot_rate', '50_100_fw_low_data_indicator', '50_100_fw_poor_shot_avoid_rate', '50_100_fw_proximity_per_shot', '50_100_fw_sg_per_shot', '50_100_fw_shot_count', 'dg_id', 'over_150_rgh_gir_rate', 'over_150_rgh_good_shot_rate', 'over_150_rgh_low_data_indicator', 'over_150_rgh_poor_shot_a

In [13]:
# Remove event_id 18 from the full combined file
full_df = read_csv(INUSE_ALL_PATH, low_memory=False)
before = len(full_df)
full_df = full_df[full_df["event_id"] != 18].copy()
write_csv(INUSE_ALL_PATH, full_df)
print(f"[FILTER] Dropped event_id=18 | rows before={before:,} | rows after={len(full_df):,}")

[FILTER] Dropped event_id=18 | rows before=316,691 | rows after=315,767


In [12]:
import pandas as pd
import numpy as np

# --- Config: update this path if the file moves ---
FILE_PATH = "/Users/joshmacbook/python_projects/GolfStats/Data/in Use/combined_roundlevel_2024_present.csv"


def add_round_positions(df):
    # Drop existing derived columns if re-running
    for col in ["to_par", "cum_to_par", "round_position"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    df = df.copy()

    # to-par per round
    df["to_par"] = df["round_score"] - df["course_par"]

    # Sort and compute cumulative to-par per player per tournament instance
    # event_id repeats across years, so event_id + season is the unique tournament key
    df = df.sort_values(["event_id", "season", "dg_id", "round_num"]).reset_index(drop=True)
    df["cum_to_par"] = df.groupby(["event_id", "season", "dg_id"])["to_par"].cumsum()

    # Fallback for events missing course_par: rank on raw cumulative score instead
    missing_par_events = df.groupby(["event_id", "season"])["course_par"].apply(lambda x: x.isna().all())
    missing_par_keys = missing_par_events[missing_par_events].index

    if len(missing_par_keys) > 0:
        mask = df.set_index(["event_id", "season"]).index.isin(missing_par_keys)
        df.loc[mask, "cum_to_par"] = (
            df[mask]
            .groupby(["event_id", "season", "dg_id"])["round_score"]
            .cumsum()
            .values
        )

    # Rank players within each tournament + round
    def rank_group(group):
        valid = group["cum_to_par"].notna()
        positions = pd.Series(np.nan, index=group.index)
        if valid.any():
            positions[valid] = (
                group.loc[valid, "cum_to_par"]
                .rank(method="min", ascending=True)
                .astype(int)
            )
        return positions

    df["round_position"] = df.groupby(
        ["event_id", "season", "round_num"], group_keys=False
    ).apply(rank_group)

    df["round_position"] = pd.array(
        df["round_position"].where(df["round_position"].isna(), df["round_position"].astype("Int64")),
        dtype="Int64",
    )

    # Add text version with "T" prefix for ties, e.g. "T5" when multiple players share position 5
    tie_counts = df.groupby(["event_id", "season", "round_num", "round_position"])["dg_id"].transform("count")
    df["round_position_text"] = df["round_position"].astype(str)
    df.loc[df["round_position"].isna(), "round_position_text"] = np.nan
    df.loc[tie_counts > 1, "round_position_text"] = "T" + df.loc[tie_counts > 1, "round_position"].astype(str)

    return df


# --- Run ---
print(f"Reading:  {FILE_PATH}")
df = pd.read_csv(FILE_PATH)
print(f"  {len(df):,} rows, {len(df.columns)} columns")

df = add_round_positions(df)

nan_count = df["round_position"].isna().sum()
if nan_count > 0:
    nan_events = df[df["round_position"].isna()]["event_name"].value_counts()
    print(f"\n  {nan_count:,} rows with NaN round_position (no scoreable data):")
    for event, count in nan_events.items():
        print(f"    {event}: {count} rows")

print(f"\nWriting:  {FILE_PATH}")
df.to_csv(FILE_PATH, index=False)
print(f"  Done. {len(df):,} rows, {len(df.columns)} columns")

Reading:  /Users/joshmacbook/python_projects/GolfStats/Data/in Use/combined_roundlevel_2024_present.csv
  80,987 rows, 41 columns


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_59995/3410638535.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(rank_group)



  464 rows with NaN round_position (no scoreable data):
    Zurich Classic of New Orleans: 464 rows

Writing:  /Users/joshmacbook/python_projects/GolfStats/Data/in Use/combined_roundlevel_2024_present.csv
  Done. 80,987 rows, 41 columns
